In [24]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
import os
import sys

import cv2

import numpy as np 

import torch
import torch.nn as nn
import torch.nn.functional as F

from box import Box
import yaml

import matplotlib.pyplot as plt

from scipy.spatial.transform import Rotation as R
import random 

import time 

root_dir = os.path.abspath('../..')
if root_dir not in sys.path:
    sys.path.append(root_dir)

from src.models.update import Update, neighbours
from src.models.graph_inference import Graph as Graph_interference
from src.models.graph_training import Graph as Graph_train
from src.models.bundle_adjustment import BundleAdjustment
from src.models.dpso import DPSO 

In [26]:
model_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/model.yaml'
sonar_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/sonar.yaml'
train_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/training.yaml'

# with open(model_config_pth, "r") as f:
#     model_config = Box(yaml.safe_load(f))

# with open(sonar_config_pth, "r") as f:
#     sonar_config = Box(yaml.safe_load(f))

# with open(train_config_pth, "r") as f:
#     train_config = Box(yaml.safe_load(f))


# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# init graph instance
# PatchGraph_i = Graph_interference(model_config, sonar_config)
# PatchGraph_t = Graph_train(model_config, sonar_config, train_config)

# # init update operater instance
# UpdateOperator_t = Update(model_config)
# UpdateOperator_i = Update(model_config)



In [27]:
# test data 

data_pth_sim = f'C:/Users/janis/Projekty/Magisterka/SonarOdometry/data/seq_1/fls/200.png'

# read frame 
frame_sim = cv2.imread(data_pth_sim, 0)

frame_sim = frame_sim.astype(np.uint8)
frame_sim = torch.from_numpy(frame_sim) # convert to torch tensor
frame_sim = frame_sim.float().unsqueeze(0).unsqueeze(0).unsqueeze(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
frames_inm_series = 5
batch_size = 2
t_0 = 1.0
dt = 0.5

frame_sim_b = torch.cat([frame_sim for _ in range(frames_inm_series)], axis = 1)
frame_sim_b = torch.cat([frame_sim_b for _ in range(batch_size)], axis = 0)


time_tensor = torch.tensor([t_0 + i*dt for i in range(frames_inm_series)], device = device, dtype = torch.float)
time_tensor = time_tensor.unsqueeze(0)
time_tensor = torch.cat([time_tensor for _ in range(batch_size)], dim = 0)


print('-'*80)
print(f'Input data format:')
print(f'simulated tensor shape: {frame_sim.shape}, data type: {frame_sim.dtype}')
print(f'simulated tensors batch shape: {frame_sim_b.shape}, data type: {frame_sim_b.dtype}')
print(f'time tensor: {time_tensor.shape}')
print('-'*80)

--------------------------------------------------------------------------------
Input data format:
simulated tensor shape: torch.Size([1, 1, 1, 800, 768]), data type: torch.float32
simulated tensors batch shape: torch.Size([2, 5, 1, 800, 768]), data type: torch.float32
time tensor: torch.Size([2, 5])
--------------------------------------------------------------------------------


In [28]:
dpso_t = DPSO(model_config_pth, sonar_config_pth, train_config_pth, mode = 'train', output_dir = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/test')

In [29]:
output = dpso_t(frame_sim_b, time_tensor)

In [30]:
print(output[1, 4, :])


tensor([-0.2582,  1.2766,  0.0000,  0.0000,  0.0000, -0.0397,  0.9992])


In [53]:
dpso_i = DPSO(model_config_pth, sonar_config_pth, train_config_pth, mode = 'inference', output_dir = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/test')

In [54]:
b, n, c, h, w = frame_sim_b.shape
print(f'Batch size: {b}, frames per batch: {n}')

for batch in range(b):
    for frame_num in range(n):


        frame = frame_sim_b[batch, frame_num, :, :, :].unsqueeze(0).unsqueeze(0)
        t = time_tensor[batch, frame_num]


        pos, time, nu = dpso_i(frame, t)
        print(f't: {time}, n: {nu}, pos: {pos}')

dpso_i.close()

Batch size: 2, frames per batch: 5
t: None, n: None, pos: None
t: None, n: None, pos: None
t: None, n: None, pos: None
t: None, n: None, pos: None
t: None, n: None, pos: None
t: 0.0, n: 6, pos: tensor([0., 0., 0., 0., 0., 0., 0.])
t: 0.0, n: 7, pos: tensor([0., 0., 0., 0., 0., 0., 0.])
t: 0.0, n: 8, pos: tensor([0., 0., 0., 0., 0., 0., 0.])
t: 0.0, n: 9, pos: tensor([0., 0., 0., 0., 0., 0., 0.])
t: 1.0, n: 10, pos: tensor([-3.7012,  1.0573,  0.0000,  0.0000,  0.0000,  0.2503,  0.9682])


In [ ]:
# # --- pipeline - training ---

# show_output_shape = False

# t0 = time.time()

# poses, coords_phi = PatchGraph_t.append(frame_sim_b, time_tensor, device)

# if show_output_shape: 
#     print("\n" + "="*50)
#     print("Step 1: PatchGraph_t.append")
#     print("-" * 50)
#     print(f"• {'poses':<25} : {poses.shape}")
#     print(f"• {'coords_phi':<25} : {coords_phi.shape}")
#     print("="*50)


# corr_t, ctx_t, source_frame_idx_t, target_frame_idx_t, patch_idx_t = PatchGraph_t.update_step(poses, coords_phi, device)

# if show_output_shape: 
#     print("\n" + "="*50)
#     print("Step 2: PatchGraph_t.update_step")
#     print("-" * 50)
#     print(f"• {'corr_t':<25} : {corr_t.shape}")
#     print(f"• {'ctx_t':<25} : {ctx_t.shape}")
#     print(f"• {'source_frame_idx_t':<25} : {source_frame_idx_t.shape}")
#     print(f"• {'target_frame_idx_t':<25} : {target_frame_idx_t.shape}")
#     print(f"• {'patch_idx_t':<25} : {patch_idx_t.shape}")
#     print("="*50)

# t1 = time.time()

# n_edges = corr_t.shape[0]
# h_init = torch.zeros((n_edges, model_config.CONTEXT_OUTPUT_CH), device=device, dtype=torch.float)

# h, correction = UpdateOperator_t(h_init, None, corr_t, ctx_t, source_frame_idx_t, target_frame_idx_t, patch_idx_t, device)
# delta, weights = correction 

# if show_output_shape: 
#     print("\n" + "="*50)
#     print("Step 3: UpdateOperator")
#     print("-" * 50)
#     print(f"• {'h (ukryty stan)':<25} : {h.shape}")
#     print(f"• {'delta (korekcja)':<25} : {delta.shape}")
#     print(f"• {'weights (wagi)':<25} : {weights.shape}")
#     print("="*50)

# t2 = time.time()

# BA = BundleAdjustment(poses, PatchGraph_t.coords_r_theta, coords_phi)

# BA.init_ba(source_frame_idx_t, patch_idx_t, delta, weights)

# opt_poses, opt_elevation = BA.run(max_iter=7, early_stop_tol=10e-3)

# if show_output_shape: 
#     print("\n" + "="*50)
#     print("Step 4: Bundle Adjustment (Wyjście)")
#     print("-" * 50)
#     print(f"• {'opt_poses':<25} : {opt_poses.shape}")
#     print(f"• {'opt_elevation':<25} : {opt_elevation.shape}")
#     print("="*50 + "\n")

# t3 = time.time()


# print('Execution time:')
# print('-' * 20)
# print(f"Graph             : {t1 - t0:.4f} s")
# print(f"Update            : {t2 - t1:.4f} s")
# print(f"Bundle adjustment : {t3 - t2:.4f} s")
# print(f"Overall           : {t3 - t0:.4f} s")

In [ ]:
# # --- pipeline - inference ---
# show_output_shape = False

# b, n, c, h, w = frame_sim_b.shape
# print(f'Batch size: {b}, frames per batch: {n}')

# for batch in range(b):
#     for frame_num in range(n):

    
#         frame = frame_sim_b[batch, frame_num, :, :, :].unsqueeze(0).unsqueeze(0)
#         t = time_tensor[batch, frame_num]

#         t0 = time.time()

#         PatchGraph_i.append(frame, t, device)

#         if batch * n + frame_num < 5: 
#             continue # skip if not init

#         if show_output_shape: 
#             print("\n" + "="*50)
#             print("Step 1: PatchGraph_t.append")
#             print("-" * 50)
#             # print(f"• {'poses':<25} : {poses.shape}")
#             # print(f"• {'coords_phi':<25} : {coords_phi.shape}")
#             print("="*50)


#         corr_i, ctx_i, source_frame_idx_i, target_frame_idx_i, patch_idx_i = PatchGraph_i.update_step(device)

#         if show_output_shape: 
#             print("\n" + "="*50)
#             print("Step 2: PatchGraph_t.update_step")
#             print("-" * 50)
#             print(f"• {'corr_t':<25} : {corr_t.shape}")
#             print(f"• {'ctx_t':<25} : {ctx_t.shape}")
#             print(f"• {'source_frame_idx_t':<25} : {source_frame_idx_t.shape}")
#             print(f"• {'target_frame_idx_t':<25} : {target_frame_idx_t.shape}")
#             print(f"• {'patch_idx_t':<25} : {patch_idx_t.shape}")
#             print("="*50)

#         t1 = time.time()

#         # --- uwaga!!! --- 
#         n_edges = corr_i.shape[0]
#         h_init = torch.zeros((n_edges, model_config.CONTEXT_OUTPUT_CH), device=device, dtype=torch.float)
#         # --- uwaga!!! --- 

#         h, correction = UpdateOperator_i(h_init, None, corr_i, ctx_i, source_frame_idx_i, target_frame_idx_i, patch_idx_i, device)
#         delta, weights = correction 

#         if show_output_shape: 
#             print("\n" + "="*50)
#             print("Step 3: UpdateOperator")
#             print("-" * 50)
#             print(f"• {'h (ukryty stan)':<25} : {h.shape}")
#             print(f"• {'delta (korekcja)':<25} : {delta.shape}")
#             print(f"• {'weights (wagi)':<25} : {weights.shape}")
#             print("="*50)

#         t2 = time.time()

#         print( PatchGraph_i.patch_state[:, :2].shape)

#         # print(f'poses: {poses.shape}, PatchGraph_i.patch_coords_r_theta: {PatchGraph_i.patch_coords_r_theta.shape}, PatchGraph_i.patch_coords_phi: {PatchGraph_i.patch_coords_phi.shape}')
#         BA = BundleAdjustment(poses, PatchGraph_i.patch_coords_r_theta, PatchGraph_i.patch_coords_phi)

#         BA.init_ba(source_frame_idx_i, patch_idx_i, delta, weights)

#         opt_poses, opt_elevation = BA.run(max_iter=7, early_stop_tol=10e-3)

#         if show_output_shape: 
#             print("\n" + "="*50)
#             print("Step 4: Bundle Adjustment (Wyjście)")
#             print("-" * 50)
#             print(f"• {'opt_poses':<25} : {opt_poses.shape}")
#             print(f"• {'opt_elevation':<25} : {opt_elevation.shape}")
#             print("="*50 + "\n")

#         t3 = time.time()


#         print('Execution time:')
#         print('-' * 20)
#         print(f"Graph             : {t1 - t0:.4f} s")
#         print(f"Update            : {t2 - t1:.4f} s")
#         print(f"Bundle adjustment : {t3 - t2:.4f} s")
#         print(f"Overall           : {t3 - t0:.4f} s")